# XGBoost

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error,r2_score

Load  Data

In [2]:
X_train  = pd.read_csv('X_train_prepared.csv')
y_train  = pd.read_csv('y_train.csv').squeeze()
X_test   = pd.read_csv('X_test_prepared.csv')
test_ids = pd.read_csv('test_ids.csv').squeeze()

CAT_COLS = ['weekday_of_release', 'season_of_release', 'lunar_phase']
NUM_COLS = [c for c in X_train.columns if c not in CAT_COLS]

print({X_train.shape},{X_test.shape})

{(61609, 83)} {(41074, 83)}


Pipeline - preprocessing with scaling and encoding

In [3]:
from sklearn.preprocessing import StandardScaler
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_COLS),
    ('cat', cat_transformer, CAT_COLS)
], remainder='drop')
base_pipeline = Pipeline([
    ('prep',  preprocessor),
    ('model', XGBRegressor(n_estimators=300, random_state=42, tree_method='hist'))
])


Baseline XGB model

In [4]:
X_trn, X_val, y_trn, y_val = train_test_split(
    X_train, y_train, test_size=0.3, random_state=42
)

base_pipeline.fit(X_trn, y_trn)
base_val_pred = base_pipeline.predict(X_val)
base_rmse = np.sqrt(mean_squared_error(y_val, base_val_pred))
base_r2   = r2_score(y_val, base_val_pred)

print(f'Baseline validation RMSE : {base_rmse:.4f}')
print(f'Baseline validation R²   : {base_r2:.4f}')

Baseline validation RMSE : 11.8256
Baseline validation R²   : 0.7005


Hyperparameter Tuning

In [19]:
param_grid = {
    'max_depth': [5, 7, 8],
    'learning_rate': [0.03, 0.05],
    'subsample': [0.8, 1.0],
    'reg_alpha': [1]
}

best_rmse = np.inf
best_params = None
best_pipeline = None

for params in ParameterGrid(param_grid):
    xgb_model = Pipeline([
        ('prep', preprocessor),
        ('model', XGBRegressor(
            n_estimators=2000,
            random_state=42,
            tree_method='hist',
            eval_metric='rmse',
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            reg_alpha=params['reg_alpha'],
            subsample=params['subsample']
        ))
    ])
    xgb_model.fit(X_trn, y_trn)
    pred = xgb_model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))

    if rmse < best_rmse:
        best_rmse = rmse
        best_params = params
        best_pipeline = xgb_model

print(f'Best validation RMSE : {best_rmse:.4f}')
print(f'Best params          : {best_params}')

Best validation RMSE : 10.4979
Best params          : {'learning_rate': 0.03, 'max_depth': 8, 'reg_alpha': 1, 'subsample': 0.8}


Best Model

In [20]:
val_pred = best_pipeline.predict(X_val)
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_mae  = mean_absolute_error(y_val, val_pred)
val_mse  = mean_squared_error(y_val, val_pred)
val_r2   = r2_score(y_val, val_pred)


print(f'Tuned validation RMSE : {val_rmse:.4f}')
print(f'Tuned validation MAE  : {val_mae:.4f}')
print(f'Tuned validation R²   : {val_r2:.4f}')


Tuned validation RMSE : 10.4979
Tuned validation MAE  : 6.2057
Tuned validation R²   : 0.7640


In [21]:
best_pipeline.fit(X_train, y_train)
y_pred = best_pipeline.predict(X_train)
train_rmse = np.sqrt(mean_squared_error(y_train, y_pred))
train_mae  = mean_absolute_error(y_train, y_pred)
train_r2   = r2_score(y_train, y_pred)


In [23]:
test_preds = np.clip(best_pipeline.predict(X_test), 0, 100)
submission = pd.DataFrame({'id': test_ids.values, 'target': test_preds})
submission.to_csv('submission_xgboost.csv', index=False)
print('Saved: submission_xgboost.csv')

submission.head()

Saved: submission_xgboost.csv


,id,target
0,25174,45.921368
1,38453,69.130180
2,29013,63.610497
3,57463,70.857391
4,51264,26.223133
